<a href="https://colab.research.google.com/github/EnesDemir143/recyclableproject/blob/enes/New_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import cv2
import imghdr

<ipython-input-1-e6baf48529f5>:3: DeprecationWarning: 'imghdr' is deprecated and slated for removal in Python 3.13
  import imghdr


In [2]:
gpus = tf.config.experimental.list_physical_devices('GPU')
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

In [3]:
tf.config.list_physical_devices('GPU')

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

print("Path to dataset files:", path)

100%|██████████| 744M/744M [00:33<00:00, 23.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8


In [5]:
import os

data_dir = '/root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset'

image_exts = ['jpeg','jpg', 'bmp', 'png']

for image_class in os.listdir(data_dir):
    for image in os.listdir(os.path.join(data_dir, image_class)):
        image_path = os.path.join(data_dir, image_class, image)
        try:
            img = cv2.imread(image_path)
            tip = imghdr.what(image_path)
            if tip not in image_exts:
                print('Image not in ext list {}'.format(image_path))
                os.remove(image_path)
        except Exception as e:
            print('Issue with image {}'.format(image_path))
            # os.remove(image_path)

Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_1433.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2784.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2779.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_2184.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_3119.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/paper/paper_1678.jpg
Image not in ext list /root/.cache/kagglehub/datasets/sumn2u/garbage-classification-v2/versions/8/garbage-dataset/plastic/plastic_2038.jpg
Image not in ext list /root/.cache/kagglehub/datase

In [6]:
import numpy as np
from matplotlib import pyplot as plt

In [18]:
train_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=(400, 400),
    label_mode='categorical',
    batch_size=64,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="training")

val_data = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    image_size=(400, 400),
    label_mode='categorical',
    batch_size=64,
    shuffle=True,
    seed=42,
    validation_split=0.2,
    subset="validation",
)


Found 19750 files belonging to 10 classes.
Using 15800 files for training.
Found 19750 files belonging to 10 classes.
Using 3950 files for validation.


In [19]:
base_model = tf.keras.applications.EfficientNetV2S(include_top=False,
                                                   weights='imagenet',
                                                   input_shape=(400, 400, 3))

In [20]:
base_model.summary()

Model: "efficientnetv2-s"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3             │ (None, 400, 400, 3)    │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ rescaling_1 (Rescaling)   │ (None, 400, 400, 3)    │              0 │ input_layer_3[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_conv (Conv2D)        │ (None, 200, 200, 24)   │            648 │ rescaling_1[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_bn                   │ (None, 200, 200, 24)   │             96 │ stem_conv[0][0]        │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ stem_activation           │ (None, 200, 200, 24)   │              0 │ stem_bn[0][0]          │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_conv      │ (None, 200, 200, 24)   │          5,184 │ stem_activation[0][0]  │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_bn        │ (None, 200, 200, 24)   │             96 │ block1a_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_project_activati… │ (None, 200, 200, 24)   │              0 │ block1a_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1a_add (Add)         │ (None, 200, 200, 24)   │              0 │ block1a_project_activ… │
│                           │                        │                │ stem_activation[0][0]  │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_conv      │ (None, 200, 200, 24)   │          5,184 │ block1a_add[0][0]      │
│ (Conv2D)                  │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_bn        │ (None, 200, 200, 24)   │             96 │ block1b_project_conv[… │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_project_activati… │ (None, 200, 200, 24)   │              0 │ block1b_project_bn[0]… │
│ (Activation)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_drop (Dropout)    │ (None, 200, 200, 24)   │              0 │ block1b_project_activ… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ block1b_add (Add)         │ (None, 200, 200, 24)   │              0 │ block1b_drop[0][0],    │
│                           │                        │                │ block1a_add[0][0]      │
├──────────────────────

 Total params: 20,331,360 (77.56 MB)

 Trainable params: 20,177,488 (76.97 MB)

 Non-trainable params: 153,872 (601.06 KB)

In [21]:
for layer in base_model.layers:
    layer.trainable = False

In [22]:
data_augmentation = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"),
                                         tf.keras.layers.RandomRotation(0.2),
                                         tf.keras.layers.RandomZoom(0.2),
                                         tf.keras.layers.RandomHeight(0.2),
                                         tf.keras.layers.RandomWidth(0.2),],
                                         name ="data_augmentation")

In [23]:
name = 'EfficientNetV2S_garbage_classification'

EfficientNetV2S_model = tf.keras.Sequential([
    tf.keras.Input(shape=(None, None, 3)),
    data_augmentation,
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(len(train_data.class_names), activation='softmax')
  ],name=name)

In [24]:
EfficientNetV2S_model.compile(loss='categorical_crossentropy',
              optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              metrics=['accuracy'])

In [25]:
EfficientNetV2S_model.summary()

Model: "EfficientNetV2S_garbage_classification"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ data_augmentation (Sequential)       │ (None, None, None, 3)       │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ efficientnetv2-s (Functional)        │ (None, None, None, 1280)    │      20,331,360 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_1           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 128)                 │         163,968 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 256)                 │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │           2,570 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 20,531,434 (78.32 MB)

 Trainable params: 199,818 (780.54 KB)

 Non-trainable params: 20,331,616 (77.56 MB)

In [26]:
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3,verbose=1)

In [27]:
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss",
                                                              factor=0.2,
                                                              patience=2,
                                                              verbose=1,
                                                              min_lr=0.00001)

check_model = tf.keras.callbacks.ModelCheckpoint('EfficientNetV2S_model.h5',
                                                   monitor="val_accuracy",
                                                   mode="max",
                                                   save_best_only=True)
callback = [early_stop, reduce_learning_rate, check_model]

In [28]:
import time
start_time = time.time()
EfficientNetV2S_history = EfficientNetV2S_model.fit(train_data,
                                                    epochs=25,
                                                    steps_per_epoch=len(train_data),
                                                    validation_data=val_data,
                                                    validation_steps=len(val_data),
                                                    callbacks=callback)

Epoch 1/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.7760 - loss: 0.7012

247/247 ━━━━━━━━━━━━━━━━━━━━ 809s 3s/step - accuracy: 0.7763 - loss: 0.7002 - val_accuracy: 0.9415 - val_loss: 0.1986 - learning_rate: 0.0010
Epoch 2/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9108 - loss: 0.2644

247/247 ━━━━━━━━━━━━━━━━━━━━ 713s 3s/step - accuracy: 0.9108 - loss: 0.2644 - val_accuracy: 0.9506 - val_loss: 0.1717 - learning_rate: 0.0010
Epoch 3/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 637s 3s/step - accuracy: 0.9258 - loss: 0.2262 - val_accuracy: 0.9489 - val_loss: 0.1696 - learning_rate: 0.0010
Epoch 4/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9320 - loss: 0.2034

247/247 ━━━━━━━━━━━━━━━━━━━━ 595s 2s/step - accuracy: 0.9320 - loss: 0.2034 - val_accuracy: 0.9509 - val_loss: 0.1569 - learning_rate: 0.0010
Epoch 5/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9298 - loss: 0.1958

247/247 ━━━━━━━━━━━━━━━━━━━━ 538s 2s/step - accuracy: 0.9298 - loss: 0.1958 - val_accuracy: 0.9552 - val_loss: 0.1443 - learning_rate: 0.0010
Epoch 6/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9418 - loss: 0.1731

247/247 ━━━━━━━━━━━━━━━━━━━━ 509s 2s/step - accuracy: 0.9418 - loss: 0.1731 - val_accuracy: 0.9600 - val_loss: 0.1337 - learning_rate: 0.0010
Epoch 7/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 493s 2s/step - accuracy: 0.9447 - loss: 0.1628 - val_accuracy: 0.9597 - val_loss: 0.1467 - learning_rate: 0.0010
Epoch 8/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9426 - loss: 0.1754
Epoch 8: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
247/247 ━━━━━━━━━━━━━━━━━━━━ 471s 2s/step - accuracy: 0.9426 - loss: 0.1754 - val_accuracy: 0.9582 - val_loss: 0.1426 - learning_rate: 0.0010
Epoch 9/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9543 - loss: 0.1388

247/247 ━━━━━━━━━━━━━━━━━━━━ 448s 2s/step - accuracy: 0.9543 - loss: 0.1388 - val_accuracy: 0.9620 - val_loss: 0.1325 - learning_rate: 2.0000e-04
Epoch 10/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 435s 2s/step - accuracy: 0.9548 - loss: 0.1350 - val_accuracy: 0.9618 - val_loss: 0.1304 - learning_rate: 2.0000e-04
Epoch 11/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9574 - loss: 0.1261

247/247 ━━━━━━━━━━━━━━━━━━━━ 416s 2s/step - accuracy: 0.9574 - loss: 0.1261 - val_accuracy: 0.9628 - val_loss: 0.1282 - learning_rate: 2.0000e-04
Epoch 12/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 423s 2s/step - accuracy: 0.9554 - loss: 0.1334 - val_accuracy: 0.9620 - val_loss: 0.1293 - learning_rate: 2.0000e-04
Epoch 13/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9561 - loss: 0.1322

247/247 ━━━━━━━━━━━━━━━━━━━━ 402s 2s/step - accuracy: 0.9561 - loss: 0.1322 - val_accuracy: 0.9638 - val_loss: 0.1266 - learning_rate: 2.0000e-04
Epoch 14/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9595 - loss: 0.1137

247/247 ━━━━━━━━━━━━━━━━━━━━ 375s 2s/step - accuracy: 0.9595 - loss: 0.1137 - val_accuracy: 0.9641 - val_loss: 0.1263 - learning_rate: 2.0000e-04
Epoch 15/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 371s 2s/step - accuracy: 0.9596 - loss: 0.1155 - val_accuracy: 0.9630 - val_loss: 0.1263 - learning_rate: 2.0000e-04
Epoch 16/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9579 - loss: 0.1203
Epoch 16: ReduceLROnPlateau reducing learning rate to 4.0000001899898055e-05.
247/247 ━━━━━━━━━━━━━━━━━━━━ 358s 1s/step - accuracy: 0.9580 - loss: 0.1202 - val_accuracy: 0.9638 - val_loss: 0.1284 - learning_rate: 2.0000e-04
Epoch 17/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 366s 1s/step - accuracy: 0.9608 - loss: 0.1164 - val_accuracy: 0.9630 - val_loss: 0.1266 - learning_rate: 4.0000e-05
Epoch 18/25
247/247 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9646 - loss: 0.1081
Epoch 18: ReduceLROnPlateau reducing learning rate to 1e-05.
247/247 ━━━━━━━━━━━━━━━━━━━━ 350s 1s/step - accuracy: 0.9646 - loss: 0.1081 - val

In [29]:
end_time = time.time()
training_time = end_time - start_time
print("Total training time: {:.2f} seconds".format(training_time))
EfficientNetV2S_model.save("EfficientNetV2S_model.h5")

Total training time: 8710.11 seconds
